# 09 — Distribution Drift in Production

## What
Distribution drift is the change in the statistical properties of the data over time. Three types are commonly distinguished: (1) *covariate shift* — P(X) changes but P(Y|X) stays the same; (2) *label shift* — P(Y) changes; (3) *concept drift* — P(Y|X) changes, the hardest to detect because labels arrive with a delay.

## Why
All production ML models degrade eventually. Covariate shift degrades model performance because the model was fitted on a distribution it no longer sees. Concept drift is existential — the world changed and the model's assumptions are simply wrong. Monitoring for drift and triggering retraining is what keeps production models accurate.

## Community context
Population Stability Index (PSI) is the industry standard for tabular feature drift — threshold of 0.1 signals mild drift, 0.25 signals major drift. The Kolmogorov-Smirnov test and Maximum Mean Discrepancy (MMD) are more statistically principled alternatives. For high-dimensional data (images, text), drift is detected in embedding space.

In [ ]:
# Distribution drift detection and automated retraining trigger
import numpy as np
from scipy import stats

def psi(expected, actual, n_bins=10):
    """
    Population Stability Index.
    PSI < 0.1: no drift
    0.1-0.25:  moderate drift
    > 0.25:    major drift -> retrain
    """
    bins = np.percentile(expected, np.linspace(0, 100, n_bins+1))
    bins[0], bins[-1] = -np.inf, np.inf
    exp_counts = np.histogram(expected, bins)[0] / len(expected)
    act_counts = np.histogram(actual, bins)[0] / len(actual)
    exp_counts = np.clip(exp_counts, 1e-6, None)
    act_counts = np.clip(act_counts, 1e-6, None)
    return float(np.sum((act_counts - exp_counts) * np.log(act_counts / exp_counts)))

def ks_drift(expected, actual, alpha=0.05):
    stat, p = stats.ks_2samp(expected, actual)
    return {'statistic': round(stat, 4), 'p_value': round(p, 6), 'drift': p < alpha}

def mmd_drift(X_ref, X_new, gamma=1.0):
    """
    Maximum Mean Discrepancy with RBF kernel.
    Large MMD -> distributions differ.
    """
    def rbf_kernel(a, b):
        diffs = a[:, None] - b[None, :]
        return np.exp(-gamma * diffs**2)
    k_xx = rbf_kernel(X_ref, X_ref).mean()
    k_yy = rbf_kernel(X_new, X_new).mean()
    k_xy = rbf_kernel(X_ref, X_new).mean()
    mmd = k_xx - 2*k_xy + k_yy
    return round(float(mmd), 6)

# Simulate: feature 'transaction_amount' drifts over months
np.random.seed(42)
reference = np.random.exponential(scale=100, size=5000)  # training distribution

scenarios = [
    ('Month 1 (stable)',    np.random.exponential(100, 1000)),
    ('Month 3 (mild drift)',np.random.exponential(130, 1000)),
    ('Month 6 (major drift)',np.random.exponential(250, 1000)),
]

print(f'=== Drift Detection for transaction_amount ===')
print(f'{"Period":<25} {"PSI":>8} {"PSI Status":<18} {"KS p-val":>10} {"KS Drift":>10} {"MMD":>10}')
for name, actual in scenarios:
    p = psi(reference, actual)
    status = 'OK' if p < 0.1 else ('MODERATE' if p < 0.25 else 'MAJOR')
    ks = ks_drift(reference, actual)
    mmd = mmd_drift(reference[:500], actual[:500])
    print(f'{name:<25} {p:>8.4f} {status:<18} {ks["p_value"]:>10.4f} {str(ks["drift"]):>10} {mmd:>10.6f}')

In [ ]:
# Drift -> retrain -> redeploy pipeline simulation
import time

class DriftRetrainPipeline:
    def __init__(self, psi_threshold=0.25):
        self.threshold = psi_threshold
        self.reference_data = None
        self.model_version = 1
        self.retrains = []

    def set_reference(self, data):
        self.reference_data = data
        print(f'Reference set: n={len(data)} mean={data.mean():.1f}')

    def check_and_retrain(self, new_data, period_label):
        drift_score = psi(self.reference_data, new_data)
        print(f'[{period_label}] PSI={drift_score:.4f}', end='')
        if drift_score > self.threshold:
            print(f' -> DRIFT DETECTED -> retraining model v{self.model_version+1}...')
            # Simulate retraining on new data
            new_mean = new_data.mean()
            time.sleep(0.01)  # simulated training time
            self.model_version += 1
            self.reference_data = new_data  # update reference
            self.retrains.append({'period': period_label, 'psi': drift_score,
                                  'new_version': self.model_version})
            print(f'  -> Deployed model v{self.model_version} (trained on {len(new_data)} samples, mean={new_mean:.1f})')
        else:
            print(' -> OK, no retraining needed')

pipeline = DriftRetrainPipeline(psi_threshold=0.25)
pipeline.set_reference(np.random.exponential(100, 5000))
np.random.seed(0)
monthly_data = [
    ('Jan', np.random.exponential(105, 1000)),
    ('Feb', np.random.exponential(110, 1000)),
    ('Mar', np.random.exponential(130, 1000)),
    ('Apr', np.random.exponential(160, 1000)),
    ('May', np.random.exponential(200, 1000)),
    ('Jun', np.random.exponential(210, 1000)),
]
print()
for label, data in monthly_data:
    pipeline.check_and_retrain(data, label)
print(f'\nTotal retrains: {len(pipeline.retrains)}  Final model version: v{pipeline.model_version}')